## Импорты и загрузка датасета

In [62]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import train_test_split

from sklearn.metrics import roc_auc_score, f1_score

from catboost import CatBoostClassifier, Pool, metrics
from lightgbm import LGBMClassifier

In [1]:
RANDOM_STATE = 111
DATASET_PATH = 'https://raw.githubusercontent.com/evgpat/edu_stepik_practical_ml/main/datasets/flight_delays_train.csv'

In [3]:
data = pd.read_csv(DATASET_PATH)

In [8]:
X = data.drop("dep_delayed_15min", axis = 1)
y = data["dep_delayed_15min"] == 'Y' ## Чтобы в столбце целевой переменной были не 'Y' и 'N', а True и False

In [9]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Month          100000 non-null  str  
 1   DayofMonth     100000 non-null  str  
 2   DayOfWeek      100000 non-null  str  
 3   DepTime        100000 non-null  int64
 4   UniqueCarrier  100000 non-null  str  
 5   Origin         100000 non-null  str  
 6   Dest           100000 non-null  str  
 7   Distance       100000 non-null  int64
dtypes: int64(2), str(6)
memory usage: 6.1 MB


In [10]:
X.describe()

,DepTime,Distance
count,100000.000000,100000.00000
mean,1341.523880,729.39716
std,476.378445,574.61686
min,1.000000,30.00000
25%,931.000000,317.00000
50%,1330.000000,575.00000
75%,1733.000000,957.00000
max,2534.000000,4962.00000


In [11]:
X.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732
1,c-4,c-20,c-3,1548,US,PIT,MCO,834
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423


## Создадим список номеров колонок с категориальными признаками для бустингов

Какой длины получился список?

In [21]:
X.dtypes

Month              str
DayofMonth         str
DayOfWeek          str
DepTime          int64
UniqueCarrier      str
Origin             str
Dest               str
Distance         int64
dtype: object

In [19]:
categorical_features_indices = np.where(X.dtypes != 'int64')[0]

In [22]:
len(categorical_features_indices)

6

## Поделим данные на трейн и тест

In [74]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

In [25]:
Xtrain.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance
41207,c-4,c-18,c-1,1457,CO,EWR,TPA,998
28283,c-11,c-1,c-2,1225,UA,DEN,BOS,1754
34619,c-6,c-16,c-5,1650,YV,IAD,CAE,401
8789,c-5,c-18,c-4,923,AA,SLC,DFW,988
38315,c-2,c-14,c-2,1839,AA,STL,SAN,1558


## Модели с параметрами по умолчанию

### CatBoost

In [37]:
model = CatBoostClassifier(
    custom_loss=['AUC'],
    random_seed=RANDOM_STATE
)

model.fit(
    Xtrain, ytrain,
    cat_features=categorical_features_indices,
    logging_level='Silent', 
    plot = True
)

pred = model.predict_proba(Xtest)[:, 1] 

roc_auc_score(ytest, pred)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0.7671857694659934

### LightGBM

In [76]:
for c in X.columns:
    col_type = X[c].dtype
    if col_type == 'str':
        Xtrain[c] = Xtrain[c].astype('category')
        Xtest[c] = Xtest[c].astype('category')

In [108]:
model = LGBMClassifier(random_state=RANDOM_STATE)

model.fit(Xtrain, ytrain)

pred = model.predict_proba(Xtest)[:,1]

roc_auc_score(ytest, pred)

[LightGBM] [Info] Number of positive: 14346, number of negative: 60654
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000601 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 75000, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.191280 -> initscore=-1.441714
[LightGBM] [Info] Start training from score -1.441714


0.7341149074685321

## Выделим дополнительную валидационную выборку

In [78]:
Xtrain_new, Xval, ytrain_new, yval = train_test_split(Xtrain, ytrain, test_size=0.25, random_state=RANDOM_STATE)

Создайте функцию objective_lgbm, в которой среди гиперпараметров

* num_leaves = trial.suggest_int("num_leaves", 10, 100)
* n_estimators = trial.suggest_int("n_estimators", 10, 1000)

подберите оптимальные, обучая **LGBM** на Xtrain_new, ytrain_new и проверяя качество (ROC-AUC) на Xval.

Используйте 30 эпох обучения Optuna.

In [99]:
def objective_LGBM(trial):
    num_leaves = trial.suggest_int("num_leaves", 10, 100)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log = True)
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)
    
    model = LGBMClassifier()
    model.fit(Xtrain_new, ytrain_new)
    pred = model.predict_proba(Xval)[:,1]
    
    score = roc_auc_score(yval, pred)
    
    return score

In [103]:
study = optuna.create_study(direction = 'maximize')
study.optimize(objective_LGBM, n_trials = 100)

[I 2026-03-16 19:16:33,703] A new study created in memory with name: no-name-6ec447ba-916b-4ea0-b887-ee3f90611c14


[LightGBM] [Info] Number of positive: 10730, number of negative: 45520
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 56250, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.190756 -> initscore=-1.445108
[LightGBM] [Info] Start training from score -1.445108


[I 2026-03-16 19:16:34,072] Trial 0 finished with value: 0.7257924524688594 and parameters: {'num_leaves': 19, 'max_depth': 6, 'learning_rate': 0.03484893612036243, 'n_estimators': 914}. Best is trial 0 with value: 0.7257924524688594.


[LightGBM] [Info] Number of positive: 10730, number of negative: 45520
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000341 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 56250, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.190756 -> initscore=-1.445108
[LightGBM] [Info] Start training from score -1.445108


[I 2026-03-16 19:16:34,426] Trial 1 finished with value: 0.7257924524688594 and parameters: {'num_leaves': 86, 'max_depth': 4, 'learning_rate': 0.3429320583073637, 'n_estimators': 657}. Best is trial 0 with value: 0.7257924524688594.


...

[I 2026-03-16 19:17:10,266] Trial 97 finished with value: 0.7257924524688594 and parameters: {'num_leaves': 69, 'max_depth': 2, 'learning_rate': 0.00031132734164658214, 'n_estimators': 295}. Best is trial 0 with value: 0.7257924524688594.


[LightGBM] [Info] Number of positive: 10730, number of negative: 45520
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000326 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 56250, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.190756 -> initscore=-1.445108
[LightGBM] [Info] Start training from score -1.445108


[I 2026-03-16 19:17:10,626] Trial 98 finished with value: 0.7257924524688594 and parameters: {'num_leaves': 98, 'max_depth': 8, 'learning_rate': 1.1910715529008453e-05, 'n_estimators': 625}. Best is trial 0 with value: 0.7257924524688594.


[LightGBM] [Info] Number of positive: 10730, number of negative: 45520
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001263 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 56250, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.190756 -> initscore=-1.445108
[LightGBM] [Info] Start training from score -1.445108


[I 2026-03-16 19:17:10,980] Trial 99 finished with value: 0.7257924524688594 and parameters: {'num_leaves': 50, 'max_depth': 4, 'learning_rate': 0.0036281505034227233, 'n_estimators': 488}. Best is trial 0 with value: 0.7257924524688594.


In [104]:
study.best_params

{'num_leaves': 19,
 'max_depth': 6,
 'learning_rate': 0.03484893612036243,
 'n_estimators': 914}

In [105]:
model = LGBMClassifier(**study.best_params) 
model.fit(Xtrain, ytrain)

pred_proba = model.predict_proba(Xtest)[:,1]

roc_auc_score(ytest, pred_proba)

[LightGBM] [Info] Number of positive: 14346, number of negative: 60654
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000464 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1095
[LightGBM] [Info] Number of data points in the train set: 75000, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.191280 -> initscore=-1.441714
[LightGBM] [Info] Start training from score -1.441714


0.735463718791334

Создайте функцию objective_Cat, в которой среди гиперпараметров

* num_leaves = trial.suggest_int("num_leaves", 10, 100)
* n_estimators = trial.suggest_int("n_estimators", 10, 1000)

подберите оптимальные, обучая **CatBoost** на Xtrain_new, ytrain_new и проверяя качество (ROC-AUC) на Xval.

Используйте 30 эпох обучения Optuna.

In [111]:
def objective_Cat(trial):
    num_leaves = trial.suggest_int("num_leaves", 10, 100)
    #max_depth = trial.suggest_int("max_depth", 2, 20)
    #learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log = True)
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)
    
    model = CatBoostClassifier(
        custom_loss=['AUC']
    )

    model.fit(
        Xtrain_new, ytrain_new,
        cat_features=categorical_features_indices,
        logging_level='Silent'
    )
    
    pred = model.predict_proba(Xval)[:,1]
    
    score = roc_auc_score(yval, pred)
    
    return score

In [112]:
study = optuna.create_study(direction = 'maximize')
study.optimize(objective_Cat, n_trials = 10)

[I 2026-03-16 19:25:12,844] A new study created in memory with name: no-name-3457514e-c043-4fb4-a39b-001fb26850de
[W 2026-03-16 19:25:35,353] Trial 0 failed with parameters: {'num_leaves': 88, 'n_estimators': 299} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "C:\Users\e.karandashova.INSIDE\AppData\Local\anaconda3\envs\py11\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\e.karandashova.INSIDE\AppData\Local\Temp\ipykernel_6756\821332798.py", line 11, in objective_Cat
    model.fit(
  File "C:\Users\e.karandashova.INSIDE\AppData\Local\anaconda3\envs\py11\Lib\site-packages\catboost\core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "C:\Users\e.karandashova.INSIDE\AppData\Local\anaconda3\envs\py11\Lib\site

KeyboardInterrupt: 

In [ ]:
study.best_params

In [ ]:
model = CatBoostClassifier(custom_loss=['AUC'], **study.best_params) 
model.fit(Xtrain, ytrain)

pred_proba = model.predict_proba(Xtest)[:,1]

roc_auc_score(ytest, pred_proba)